# P2-2 · Bài 2 · Train mạng thật

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

## Lý thuyết

In [ ]:
import torch, torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
# CIFAR-10 thật (Colab, cần mạng): torchvision.datasets.CIFAR10("data", train=True, download=True, transform=T.ToTensor())
# ở đây dùng FakeData: ảnh ngẫu nhiên cùng định dạng, không cần tải
train_ds = torchvision.datasets.FakeData(size=2000, image_size=(3, 32, 32), num_classes=10, transform=T.ToTensor())
val_ds = torchvision.datasets.FakeData(size=500, image_size=(3, 32, 32), num_classes=10, transform=T.ToTensor(), random_offset=1)

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True, num_workers=2, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=256, shuffle=False)
for x, y in train_loader:      # mỗi vòng lặp trả về một batch [128, 3, 32, 32] và [128]
    print(x.shape, y.shape, len(train_loader), "batch mỗi epoch")
    break

torch.Size([128, 3, 32, 32]) torch.Size([128]) 16 batch mỗi epoch


In [ ]:
import torch.nn as nn
model, epochs = nn.Linear(10, 2), 30          # model nhỏ chỉ để xem lịch learning rate
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-4, total_steps=len(train_loader) * epochs, pct_start=0.05)
lrs = []
for step in range(len(train_loader) * epochs):
    opt.step()                                # trong vòng lặp thật: backward() rồi opt.step()
    sched.step()                              # ... và gọi sched.step() ngay sau opt.step()
    lrs.append(sched.get_last_lr()[0])
peak = max(lrs)
print(f"{len(lrs)} bước: lr bắt đầu {lrs[0]:.1e}, đạt đỉnh {peak:.1e} ở bước {lrs.index(peak)}, kết thúc {lrs[-1]:.1e}")

480 bước: lr bắt đầu 1.3e-05, đạt đỉnh 3.0e-04 ở bước 22, kết thúc 4.8e-09


In [ ]:
import torch, torch.nn as nn
def block(cin, cout):
    return nn.Sequential(nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                         nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
                         nn.MaxPool2d(2))
model = nn.Sequential(block(3, 32), block(32, 64), block(64, 128),     # 32→16→8→4
                      nn.Flatten(), nn.Linear(128 * 4 * 4, 10))
print(model(torch.randn(8, 3, 32, 32)).shape, sum(p.numel() for p in model.parameters()), "tham số")

torch.Size([8, 10]) 308394 tham số


In [ ]:
import torchvision.transforms as T
train_tf = T.Compose([
    T.RandomCrop(32, padding=4),            # đệm 4 pixel rồi cắt ngẫu nhiên lại 32×32
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.2),
    T.ToTensor(),
    T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)),
])
test_tf = T.Compose([T.ToTensor(), T.Normalize((0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616))])
img = T.ToPILImage()(torch.rand(3, 32, 32))           # một ảnh thử
a, b = train_tf(img), train_tf(img)                   # cùng ảnh, hai lần biến đổi ngẫu nhiên khác nhau
print(a.shape, "giống nhau?", torch.equal(a, b))

torch.Size([3, 32, 32]) giống nhau? False
